# Bitcoin Quantitative Backtesting & Systematic Strategy Validation

This research notebook provides an institutional-grade, reproducible simulation and benchmarking of systematic Bitcoin investment strategies across historical market regimes.

### Benchmarked Strategies:
1. **Dynamic Reserve DCA + Macro Regime Overlay**: 70% Base DCA Pool + 30% Tactical Reserve Pool with dynamic multipliers (0x to 2.0x + 25% tactical reserve draw) driven by Mayer Multiple, MVRV, Fear & Greed sentiment, and high-impact macro circuit breakers.
2. **Blind Dollar-Cost Averaging (Blind DCA)**: Naive periodic dollar-cost averaging executing unconditionally on schedule.
3. **Lump Sum Buy & Hold**: Full initial capital deployment at inception ($T_0$) with zero subsequent contributions.

### Core Analytical Mart:
- `mart_btc_investment_signals_daily`

In [ ]:
from datetime import date
from pathlib import Path

from bitcoin_data_platform.backtest.engine import BacktestEngine
from bitcoin_data_platform.backtest.models import (
    BacktestConfig,
    BacktestDayRecord,
    FrequencyType,
    StrategyType,
)
from bitcoin_data_platform.backtest.reporter import (
    format_table,
)

print("Backtest Engine components imported successfully.")

## 1. Load Conformed Historical Data

Connect to the local DuckDB database or generate synthetic backtest cycles if running in an isolated clean-room environment.

In [ ]:
repo_root = (
    Path("..").resolve() if (Path("..") / "pyproject.toml").exists() else Path(".").resolve()
)
db_path = repo_root / "data" / "state" / "platform.duckdb"
engine = BacktestEngine()

records = []
if db_path.exists():
    try:
        records = engine.load_data_from_duckdb(db_path=db_path)
        print(f"Loaded {len(records)} daily records from DuckDB database.")
    except Exception as exc:
        print(f"Could not load from DuckDB ({exc}); initializing synthetic cycle dataset.")

if not records:
    # Synthetic 2-year market cycle for clean-room demonstration
    from datetime import timedelta

    start_dt = date(2024, 1, 1)
    price = 42000.0
    for i in range(730):
        d = start_dt + timedelta(days=i)
        # Simulate cycle: accumulation -> bull run -> correction
        if i < 180:
            price += 50.0 + (i % 7) * 20.0
            signal = "OPPORTUNISTIC_ACCUMULATE" if i % 10 == 0 else "STANDARD_DCA"
            macro = i % 30 == 0
        elif i < 400:
            price += 150.0 + (i % 5) * 30.0
            signal = "HARD_FREEZE" if price > 85000 else "DEFENSIVE_RESERVE"
            macro = False
        elif i < 550:
            price -= 120.0 + (i % 3) * 40.0
            signal = "AGGRESSIVE_ACCUMULATE" if price < 60000 else "OPPORTUNISTIC_ACCUMULATE"
            macro = i % 25 == 0
        else:
            price += 80.0
            signal = "STANDARD_DCA"
            macro = False
        records.append(
            BacktestDayRecord(
                trade_date=d,
                market_close_usd=round(price, 2),
                sma_200=round(price * 0.95, 2),
                mayer_multiple=round(price / (price * 0.95), 2),
                mvrv_ratio=1.8,
                fng_value=55,
                has_high_impact_macro_event=macro,
                investment_signal=signal,
            )
        )
    print(f"Generated {len(records)} days of synthetic market cycle records for simulation.")

## 2. Configure Simulation Parameters

Set institutional parameters:
- Initial cash: $10,000 USD (for Lump Sum benchmark)
- Periodic DCA contribution: $100 USD daily
- Trading execution fee: 10.0 basis points (0.10% Coinbase spot tier)
- Risk-free benchmark rate: 3.0% annualized

In [ ]:
config = BacktestConfig(
    initial_cash=10000.0,
    periodic_amount=100.0,
    frequency=FrequencyType.DAILY,
    fee_bps=10.0,
    risk_free_rate=0.03,
)
print("BacktestConfig configured:", config)

## 3. Execute Multi-Strategy Benchmark

Run all three strategies across the identical dataset and chronological event loop without lookahead bias.

In [ ]:
summary = engine.run_benchmark(records=records, config=config)
print(
    f"Completed simulation for {summary.duration_days} days "
    f"across {len(summary.results)} strategies."
)

## 4. Performance Benchmark Results

In [ ]:
print(format_table(summary))

## 5. Tactical Reserve Pool Dynamics & Analysis

In [ ]:
dynamic_res = summary.results[StrategyType.DYNAMIC_RESERVE]
blind_res = summary.results[StrategyType.BLIND_DCA]
lump_res = summary.results[StrategyType.LUMP_SUM]

print("=== Quantitative Risk-Adjusted Analysis ===")
print(f"Dynamic Reserve DCA Max Drawdown : {dynamic_res.max_drawdown_pct:.2f}%")
print(f"Blind DCA Max Drawdown           : {blind_res.max_drawdown_pct:.2f}%")
print(f"Lump Sum Max Drawdown            : {lump_res.max_drawdown_pct:.2f}%")
print()
print(f"Dynamic Reserve Sharpe Ratio     : {dynamic_res.sharpe_ratio:.2f}")
print(f"Blind DCA Sharpe Ratio           : {blind_res.sharpe_ratio:.2f}")
print(f"Lump Sum Sharpe Ratio            : {lump_res.sharpe_ratio:.2f}")
print()
print(f"Dynamic Reserve Acquisition Disc : {dynamic_res.acquisition_discount_pct:+.2f}%")
print(f"Blind DCA Acquisition Discount   : {blind_res.acquisition_discount_pct:+.2f}%")
print(f"Tactical Reserve Peak Capital    : ${dynamic_res.reserve_pool_peak:,.2f}")
print(f"Tactical Reserve Final Capital   : ${dynamic_res.reserve_pool_final:,.2f}")